
# Revision 06 — Selected spatial MS-AA archetypes: interpretation and subject support

This notebook follows directly from the cross-fitted top-\(m\) analysis.

**Locked choices**
- Spatial MS-AA, across-condition shared basis
- Representative \(K=46\)
- Primary sparse subset: top-5
- Selected archetypes: **[4, 5, 6, 12, 8]**

These IDs are treated as preselected from the held-out analysis and are not changed based on appearance.

The notebook asks three questions:

1. **What is each archetype?** Show the mean temporal motif `sXC[:, k]`, the pooled participant-averaged spatial coefficient map `S[k, :]`, and Yeo-7 composition.
2. **Is each motif broadly supported across participants?** Quantify the participant-level coefficient magnitude \(\|S^{(s)}_{k,:}\|_2\), the exact rank-1 reconstruction magnitude \(\|sXC^{(s)}_{:,k}\|_2\|S^{(s)}_{k,:}\|_2\), and leave-one-out spatial-map similarity.
3. **Does expression differ by condition?** Compare intact, word-scrambled, and rest as **independent participant groups**.

### Important correction from the previous notebook

The experimental design contains **108 participants total, 36 per condition**, rather than 36 participants measured in all three conditions. Therefore intact, word, and rest must **not** be paired by within-condition row number. This notebook treats all 108 fit entries as independent participants and uses independent-group condition tests.

For spatial AA, each node's archetype coefficient vector is simplex-constrained. Consequently, `support_l2` is best interpreted as the magnitude/concentration of the spatial allocation to archetype \(k\), not as an unconstrained loading amplitude. The rank-1 Frobenius magnitude is included as a complementary contribution metric.


In [ ]:

from pathlib import Path
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import friedmanchisquare, wilcoxon

REPRESENTATIVE_K = 46
SELECTED_ARCHETYPES = [4, 5, 6, 12, 8]

FIT_LOAD_DIR = Path("msaa_flexible_outputs_npz")
FIT_FILE = FIT_LOAD_DIR / f"spatialAA_across_acrossCond_K{REPRESENTATIVE_K}.npz"

OUTPUT_DIR = Path("revision_05_archetype_interpretation_and_support")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONDITION_ORDER = ["intact", "word", "rest"]
SUPPORT_METRIC = "support_l2"  # requested primary spatial-support metric
SPATIAL_DISPLAY_THRESHOLD_FRAC = 0.20

NODE_CODES_CSV = None
SCHAEFER_TXT = Path("data/pieman/raw/Schaefer2018_1000Parcels_7Networks_order.txt")
SCHAEFER_NII = Path("data/pieman/raw/Schaefer2018_1000Parcels_7Networks_order_FSLMNI152_2mm.nii.gz")
POSTERIOR_MAT = Path("data/pieman/raw/pieman_posterior_K700.mat")

NETWORK_ORDER = [
    "Visual",
    "Somatomotor",
    "Dorsal attention",
    "Ventral attention",
    "Limbic",
    "Frontoparietal",
    "Default mode",
]

print("Fit file:", FIT_FILE)
print("Representative K:", REPRESENTATIVE_K)
print("Selected archetypes:", SELECTED_ARCHETYPES)


## 1. Load the locked fit

In [ ]:

def to_float_array(x):
    return np.asarray(x, dtype=float)

def load_msaa_npz(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing fit file: {path}")

    data = np.load(path, allow_pickle=True)
    results_subj = data["results_subj"].tolist()
    if isinstance(results_subj, np.ndarray):
        results_subj = results_subj.tolist()

    out = {
        "K": int(np.asarray(data["K"]).item()),
        "results_subj": results_subj,
        "condition_labels_str": np.asarray(data["condition_labels_str"]).astype(str),
    }

    for key in [
        "condition_codes", "condition_names", "analysis_type", "fit_scope",
        "centers", "widths"
    ]:
        if key in data:
            out[key] = data[key]

    return out

fit = load_msaa_npz(FIT_FILE)
results_subj = fit["results_subj"]
condition_labels = np.asarray(fit["condition_labels_str"])

assert fit["K"] == REPRESENTATIVE_K
assert max(SELECTED_ARCHETYPES) < REPRESENTATIVE_K

print("N fit entries:", len(results_subj))
display(pd.Series(condition_labels).value_counts().to_frame("count"))

example = results_subj[0]
print("subj0 sXC shape:", to_float_array(example["sXC"]).shape)
print("subj0 S shape:", to_float_array(example["S"]).shape)



For spatial AA:
- `sXC` has shape \(T\times K\)
- `S` has shape \(K\times V\)
- reconstruction is `sXC @ S`

Therefore `sXC[:, k]` is the temporal motif and `S[k, :]` is the observation-specific spatial coefficient vector.


## 2. Node coordinates and Yeo-7 labels

In [ ]:

if "centers" in fit and "widths" in fit:
    centers = np.asarray(fit["centers"], dtype=float)
    widths = np.asarray(fit["widths"], dtype=float).ravel()
else:
    from scipy.io import loadmat
    if not POSTERIOR_MAT.exists():
        raise FileNotFoundError(
            "Fit lacks centers/widths and posterior file is missing: "
            f"{POSTERIOR_MAT}"
        )
    posterior = loadmat(POSTERIOR_MAT)
    centers = np.asarray(
        posterior["posterior"]["centers"][0][0][0][0][0],
        dtype=float,
    )
    widths = np.asarray(
        list(posterior["posterior"]["widths"][0][0][0][0][0][:, 0].T),
        dtype=float,
    ).ravel()

V = to_float_array(results_subj[0]["S"]).shape[1]
assert centers.shape[0] == V
assert widths.shape[0] == V

print("centers:", centers.shape)
print("widths:", widths.shape)


In [ ]:

LOOKUP_TABLE = {
    "Vis": "Visual",
    "SomMot": "Somatomotor",
    "DorsAttn": "Dorsal attention",
    "SalVentAttn": "Ventral attention",
    "Limbic": "Limbic",
    "Cont": "Frontoparietal",
    "Default": "Default mode",
}
NETWORK_CODES = {name: i + 1 for i, name in enumerate(NETWORK_ORDER)}

def build_node_codes(centers, widths, schaefer_txt, schaefer_nii):
    try:
        import nibabel as nib
        try:
            from nilearn.maskers import NiftiMasker
        except ImportError:
            from nilearn.input_data import NiftiMasker
    except ImportError as exc:
        raise ImportError(
            "nibabel + nilearn are required to rebuild the Yeo mapping. "
            "Alternatively set NODE_CODES_CSV to a precomputed mapping."
        ) from exc

    schaefer_txt = Path(schaefer_txt)
    schaefer_nii = Path(schaefer_nii)

    if not schaefer_txt.exists():
        raise FileNotFoundError(schaefer_txt)
    if not schaefer_nii.exists():
        raise FileNotFoundError(schaefer_nii)

    def fullfact(dims):
        vals = np.asmatrix(range(1, dims[0] + 1)).T
        if len(dims) == 1:
            return vals
        aftervals = np.asmatrix(fullfact(dims[1:]))
        inds = np.asmatrix(np.zeros((np.prod(dims), len(dims))))
        row = 0
        for i in range(aftervals.shape[0]):
            inds[row:(row + len(vals)), 0] = vals
            inds[row:(row + len(vals)), 1:] = np.tile(aftervals[i, :], (len(vals), 1))
            row += len(vals)
        return inds

    def nii2cmu(nifti_file):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            img = nib.load(str(nifti_file))
            masker = NiftiMasker(mask_strategy="background")
            masker.fit(str(nifti_file))

        saff = img.get_sform()
        y = np.float32(masker.transform(str(nifti_file))).copy()
        mask_flat = np.reshape(
            masker.mask_img_.dataobj,
            (1, np.prod(masker.mask_img_.shape)),
            order="C",
        )
        vmask = np.nonzero(np.asarray(mask_flat))[1]
        vox_coords = fullfact(img.shape[0:3])[vmask, ::-1] - 1
        r = np.asarray(vox_coords @ saff[0:3, 0:3]) + saff[:3, 3]
        return {"Y": y, "R": r}

    def rbf(r, center, width):
        return np.exp(-np.sum((r - center) ** 2, axis=1) / width)

    key = pd.read_csv(
        schaefer_txt,
        sep="\t",
        header=None,
        names=["id", "name", "x", "y", "z", "t"],
    ).drop("t", axis=1)

    key["network"] = key["name"].apply(
        lambda x: LOOKUP_TABLE[x.split("_")[2]]
    )
    key["code"] = key["network"].map(NETWORK_CODES)
    key = key.set_index("id")
    key.loc[0, "code"] = 0

    networks_cmu = nii2cmu(schaefer_nii)
    networks_cmu["Y"] = np.atleast_2d(
        np.asarray(
            [key.loc[i, "code"] for i in networks_cmu["Y"].ravel()],
            dtype=float,
        )
    )

    labels = []
    for center, width in zip(centers, widths):
        overlap = rbf(networks_cmu["R"], center, width)
        label_weights = [
            np.sum(overlap[networks_cmu["Y"].ravel() == code])
            for code in range(1, len(NETWORK_ORDER) + 1)
        ]
        labels.append(int(np.argmax(label_weights) + 1))

    return pd.DataFrame({
        "code": labels,
        "Network": [NETWORK_ORDER[code - 1] for code in labels],
    })

if NODE_CODES_CSV is not None and Path(NODE_CODES_CSV).exists():
    node_codes = pd.read_csv(NODE_CODES_CSV)
else:
    node_codes = build_node_codes(
        centers, widths, SCHAEFER_TXT, SCHAEFER_NII
    )

assert len(node_codes) == V
display(
    node_codes["Network"]
    .value_counts()
    .reindex(NETWORK_ORDER)
    .to_frame("n_nodes")
)


## 3. What does each selected archetype look like?

In [ ]:

def get_temporal_matrix(results_subj, k):
    return np.stack(
        [to_float_array(sub["sXC"])[:, k] for sub in results_subj],
        axis=0,
    )

def get_spatial_matrix(results_subj, k):
    return np.stack(
        [to_float_array(sub["S"])[k, :] for sub in results_subj],
        axis=0,
    )

def sem(x, axis=0):
    x = np.asarray(x, dtype=float)
    n = np.sum(np.isfinite(x), axis=axis)
    return np.nanstd(x, axis=axis, ddof=1) / np.sqrt(np.maximum(n, 1))

group_temporal = {}
group_spatial = {}
network_rows = []

for k in SELECTED_ARCHETYPES:
    temporal = get_temporal_matrix(results_subj, k)
    spatial = get_spatial_matrix(results_subj, k)

    group_temporal[k] = {
        "mean": np.nanmean(temporal, axis=0),
        "sem": sem(temporal, axis=0),
    }

    group_map = np.nanmean(spatial, axis=0)
    group_spatial[k] = group_map

    abs_weights = np.abs(group_map)
    total = abs_weights.sum()

    for network in NETWORK_ORDER:
        mask = node_codes["Network"].to_numpy() == network
        mass = abs_weights[mask].sum() / total if total > 0 else np.nan
        network_rows.append({
            "K": REPRESENTATIVE_K,
            "archetype": k,
            "network": network,
            "network_mass": mass,
        })

network_mass_df = pd.DataFrame(network_rows)
display(network_mass_df.head(14))



### Main interpretation figure
Each row contains:
1. the mean temporal motif ± SEM;
2. the subject/condition-averaged spatial coefficient map, shown as an MNI \(x\)-\(z\) projection with node size proportional to coefficient magnitude;
3. the Yeo-7 composition calculated from **all 700 nodes** as the fraction of absolute group-map weight in each network.

The node threshold affects display only; it does not affect network composition.


In [ ]:

n_arch = len(SELECTED_ARCHETYPES)
fig, axes = plt.subplots(
    n_arch, 3,
    figsize=(15, 2.8 * n_arch),
)

if n_arch == 1:
    axes = axes[None, :]

network_categories = {name: i for i, name in enumerate(NETWORK_ORDER)}

for row, k in enumerate(SELECTED_ARCHETYPES):
    # A. temporal motif
    ax = axes[row, 0]
    mean_tc = group_temporal[k]["mean"]
    sem_tc = group_temporal[k]["sem"]
    t = np.arange(len(mean_tc))
    ax.plot(t, mean_tc, linewidth=1.8)
    ax.fill_between(t, mean_tc - sem_tc, mean_tc + sem_tc, alpha=0.2)
    ax.axhline(0, linewidth=0.6, linestyle=":")
    ax.set_ylabel(f"A{k}\ncoefficient")
    if row == 0:
        ax.set_title("Temporal motif")
    if row == n_arch - 1:
        ax.set_xlabel("Timepoint")

    # B. spatial map
    ax = axes[row, 1]
    weights = np.asarray(group_spatial[k], dtype=float)
    abs_w = np.abs(weights)
    max_w = abs_w.max()

    keep = (
        abs_w >= SPATIAL_DISPLAY_THRESHOLD_FRAC * max_w
        if max_w > 0
        else np.ones(len(weights), dtype=bool)
    )

    selected_abs = abs_w[keep]
    marker_sizes = (
        15 + 180 * selected_abs / selected_abs.max()
        if len(selected_abs) and selected_abs.max() > 0
        else np.full(keep.sum(), 20.0)
    )

    codes = (
        node_codes.loc[keep, "Network"]
        .map(network_categories)
        .to_numpy()
    )

    ax.scatter(
        centers[keep, 0],
        centers[keep, 2],
        s=marker_sizes,
        c=codes,
        alpha=0.65,
    )
    ax.axvline(0, linewidth=0.5, linestyle=":")
    ax.set_aspect("equal", adjustable="datalim")
    ax.set_xlabel("MNI x")
    ax.set_ylabel("MNI z")
    if row == 0:
        ax.set_title("Mean spatial coefficients")

    # C. Yeo-7 composition
    ax = axes[row, 2]
    cur = (
        network_mass_df[network_mass_df["archetype"] == k]
        .set_index("network")
        .reindex(NETWORK_ORDER)
    )
    ax.barh(np.arange(len(NETWORK_ORDER)), cur["network_mass"].to_numpy())
    ax.set_yticks(np.arange(len(NETWORK_ORDER)))
    ax.set_yticklabels(NETWORK_ORDER)
    ax.invert_yaxis()
    if row == 0:
        ax.set_title("Yeo-7 weight composition")
    if row == n_arch - 1:
        ax.set_xlabel("Fraction of |group map weight|")

fig.suptitle(
    f"Selected spatial MS-AA archetypes (K={REPRESENTATIVE_K})",
    y=1.01,
)
fig.tight_layout()

main_fig_path = OUTPUT_DIR / "selected_archetypes_main.png"
fig.savefig(main_fig_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", main_fig_path)


## 4. Is each motif broadly supported across participants?

In [ ]:

# ============================================================
# PARTICIPANT-LEVEL SUPPORT TABLE
# ============================================================

rows = []

for participant_index, (sub, condition) in enumerate(
    zip(results_subj, condition_labels)
):
    condition = str(condition)
    sXC = to_float_array(sub["sXC"])
    S = to_float_array(sub["S"])

    full_S_fro = np.linalg.norm(S, ord="fro")

    for k in SELECTED_ARCHETYPES:
        spatial_vec = S[k, :]
        temporal_vec = sXC[:, k]

        support_l1 = np.sum(np.abs(spatial_vec))
        support_l2 = np.linalg.norm(spatial_vec, ord=2)
        temporal_l2 = np.linalg.norm(temporal_vec, ord=2)

        # Exact Frobenius norm of the rank-1 archetype contribution:
        # || temporal[:,None] @ spatial[None,:] ||_F
        component_fro = temporal_l2 * support_l2

        support_relative = (
            support_l2 / full_S_fro
            if full_S_fro > 0
            else np.nan
        )

        rows.append({
            "participant_index": participant_index,
            "condition": condition,
            "K": REPRESENTATIVE_K,
            "archetype": k,
            "support_l1": support_l1,
            "support_l2": support_l2,
            "temporal_l2": temporal_l2,
            "component_fro": component_fro,
            "support_relative": support_relative,
        })

support_df = pd.DataFrame(rows)

print("Unique participants:", support_df["participant_index"].nunique())
print("Condition counts:")
display(
    support_df[["participant_index", "condition"]]
    .drop_duplicates()["condition"]
    .value_counts()
    .reindex(CONDITION_ORDER)
    .to_frame("n")
)

display(support_df.head(12))



### Spatial-map consistency

Magnitude alone does not establish that the pooled spatial map is representative. For each participant and selected archetype, we therefore compare that participant's `S[k, :]` map with a **leave-one-out mean map from other participants in the same condition** using cosine similarity.

A high within-condition leave-one-out similarity indicates that the group map is not being created by averaging incompatible participant-specific patterns.


In [ ]:

def cosine_similarity(a, b, eps=1e-12):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom <= eps:
        return np.nan
    return float(np.dot(a, b) / denom)

map_similarity_rows = []

for condition in CONDITION_ORDER:
    idx = np.where(condition_labels == condition)[0]

    for k in SELECTED_ARCHETYPES:
        maps = np.stack(
            [to_float_array(results_subj[i]["S"])[k, :] for i in idx],
            axis=0,
        )

        total_map = maps.sum(axis=0)

        for local_i, global_i in enumerate(idx):
            if len(idx) > 1:
                loo_mean = (total_map - maps[local_i]) / (len(idx) - 1)
            else:
                loo_mean = np.full(maps.shape[1], np.nan)

            sim = cosine_similarity(maps[local_i], loo_mean)

            map_similarity_rows.append({
                "participant_index": int(global_i),
                "condition": condition,
                "archetype": k,
                "loo_map_cosine": sim,
            })

map_similarity_df = pd.DataFrame(map_similarity_rows)

support_df = support_df.merge(
    map_similarity_df,
    on=["participant_index", "condition", "archetype"],
    how="left",
)

map_similarity_summary = (
    support_df
    .groupby(["archetype", "condition"], as_index=False)
    .agg(
        mean_loo_cosine=("loo_map_cosine", "mean"),
        median_loo_cosine=("loo_map_cosine", "median"),
        sd_loo_cosine=("loo_map_cosine", "std"),
        n=("loo_map_cosine", "size"),
    )
)

display(map_similarity_summary)


In [ ]:

# ============================================================
# SUPPORT CONCENTRATION: OVERALL + WITHIN CONDITION
# ============================================================

def effective_contributor_summary(values):
    vals = np.asarray(values, dtype=float)
    vals = np.nan_to_num(vals, nan=0.0, posinf=0.0, neginf=0.0)
    vals = np.abs(vals)

    total = vals.sum()
    n = len(vals)

    if total <= 0 or n == 0:
        return {
            "n": n,
            "max_share": np.nan,
            "effective_n": np.nan,
            "effective_fraction": np.nan,
        }

    shares = vals / total
    effective_n = 1.0 / np.sum(shares ** 2)

    return {
        "n": n,
        "max_share": float(shares.max()),
        "effective_n": float(effective_n),
        "effective_fraction": float(effective_n / n),
    }

summary_rows = []

for k in SELECTED_ARCHETYPES:
    # Overall across all 108 independent participants
    cur = support_df[support_df["archetype"] == k]
    stats = effective_contributor_summary(cur[SUPPORT_METRIC])
    summary_rows.append({
        "archetype": k,
        "condition": "ALL",
        "support_metric": SUPPORT_METRIC,
        "mean_support": cur[SUPPORT_METRIC].mean(),
        "median_support": cur[SUPPORT_METRIC].median(),
        "sd_support": cur[SUPPORT_METRIC].std(),
        "mean_loo_cosine": cur["loo_map_cosine"].mean(),
        **stats,
    })

    # Within each independent condition group
    for condition in CONDITION_ORDER:
        cc = cur[cur["condition"] == condition]
        stats = effective_contributor_summary(cc[SUPPORT_METRIC])
        summary_rows.append({
            "archetype": k,
            "condition": condition,
            "support_metric": SUPPORT_METRIC,
            "mean_support": cc[SUPPORT_METRIC].mean(),
            "median_support": cc[SUPPORT_METRIC].median(),
            "sd_support": cc[SUPPORT_METRIC].std(),
            "mean_loo_cosine": cc["loo_map_cosine"].mean(),
            **stats,
        })

support_summary_df = pd.DataFrame(summary_rows)

display(
    support_summary_df[
        support_summary_df["condition"] == "ALL"
    ].reset_index(drop=True)
)

print("Within-condition support breadth:")
display(
    support_summary_df[
        support_summary_df["condition"] != "ALL"
    ].reset_index(drop=True)
)



For each archetype, participant support values are converted into nonnegative shares \(p_s\), with

\[
N_{\mathrm{eff}} = \frac{1}{\sum_s p_s^2}.
\]

`effective_fraction = N_eff / N`. Values near 1 indicate that support is distributed broadly rather than being dominated by a few participants. The **within-condition** rows are the cleanest check because they do not mix experimental groups.


In [ ]:

# ============================================================
# FIGURE: SUPPORT ACROSS ALL 108 PARTICIPANTS
# ============================================================

fig, ax = plt.subplots(figsize=(8, 5))
rng = np.random.default_rng(42)
positions = np.arange(len(SELECTED_ARCHETYPES))
data_for_box = []

for pos, k in zip(positions, SELECTED_ARCHETYPES):
    vals = (
        support_df.loc[support_df["archetype"] == k, SUPPORT_METRIC]
        .to_numpy(dtype=float)
    )
    data_for_box.append(vals)
    jitter = rng.normal(0, 0.045, size=len(vals))
    ax.scatter(
        np.full(len(vals), pos) + jitter,
        vals,
        alpha=0.35,
        s=18,
    )

ax.boxplot(
    data_for_box,
    positions=positions,
    widths=0.45,
    showfliers=False,
)
ax.set_xticks(positions)
ax.set_xticklabels([f"A{k}" for k in SELECTED_ARCHETYPES])
ax.set_xlabel("Selected archetype")
ax.set_ylabel(SUPPORT_METRIC)
ax.set_title(
    f"Participant-level spatial support across 108 independent participants (K={REPRESENTATIVE_K})"
)
fig.tight_layout()

support_fig_path = OUTPUT_DIR / "participant_support_distribution_independent.png"
fig.savefig(support_fig_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", support_fig_path)


## 5. Does expression/support differ by condition?

In [ ]:

condition_summary_df = (
    support_df
    .groupby(["archetype", "condition"], as_index=False)
    .agg(
        mean_support=(SUPPORT_METRIC, "mean"),
        median_support=(SUPPORT_METRIC, "median"),
        sd_support=(SUPPORT_METRIC, "std"),
        n=(SUPPORT_METRIC, "size"),
        mean_component_fro=("component_fro", "mean"),
        mean_relative=("support_relative", "mean"),
        mean_loo_cosine=("loo_map_cosine", "mean"),
    )
)

condition_summary_df["sem_support"] = (
    condition_summary_df["sd_support"]
    / np.sqrt(condition_summary_df["n"])
)

condition_summary_df["condition_share"] = (
    condition_summary_df["mean_support"]
    / condition_summary_df.groupby("archetype")["mean_support"].transform("sum")
)

display(condition_summary_df.sort_values(["archetype", "condition"]))


In [ ]:

fig, axes = plt.subplots(
    1,
    len(SELECTED_ARCHETYPES),
    figsize=(3.2 * len(SELECTED_ARCHETYPES), 4.8),
    sharey=False,
)

if len(SELECTED_ARCHETYPES) == 1:
    axes = [axes]

rng = np.random.default_rng(43)

for ax, k in zip(axes, SELECTED_ARCHETYPES):
    cur = support_df[support_df["archetype"] == k]
    box_data = []

    for i, condition in enumerate(CONDITION_ORDER):
        vals = cur.loc[cur["condition"] == condition, SUPPORT_METRIC].to_numpy(dtype=float)
        box_data.append(vals)
        jitter = rng.normal(0, 0.045, size=len(vals))
        ax.scatter(
            np.full(len(vals), i) + jitter,
            vals,
            alpha=0.45,
            s=18,
        )

    ax.boxplot(
        box_data,
        positions=np.arange(len(CONDITION_ORDER)),
        widths=0.5,
        showfliers=False,
    )
    ax.set_xticks(np.arange(len(CONDITION_ORDER)))
    ax.set_xticklabels(CONDITION_ORDER, rotation=35, ha="right")
    ax.set_title(f"A{k}")
    ax.set_xlabel("Condition")

axes[0].set_ylabel(SUPPORT_METRIC)
fig.suptitle(
    f"Condition-specific spatial support: independent groups (K={REPRESENTATIVE_K})",
    y=1.02,
)
fig.tight_layout()

condition_fig_path = OUTPUT_DIR / "condition_specific_support_independent.png"
fig.savefig(condition_fig_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", condition_fig_path)



### Independent-group condition tests

The three conditions contain different participants. For each selected archetype, the primary omnibus comparison is therefore a **Kruskal-Wallis test** across intact, word, and rest. Pairwise follow-ups use two-sided **Mann-Whitney U** tests with Holm correction across the three contrasts within each archetype. The five omnibus tests are also Holm-corrected across archetypes.


In [ ]:

from scipy.stats import kruskal, mannwhitneyu


def holm_adjust(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    m = len(pvalues)
    order = np.argsort(pvalues)
    adjusted = np.empty(m, dtype=float)
    running_max = 0.0

    for rank, idx in enumerate(order):
        candidate = (m - rank) * pvalues[idx]
        running_max = max(running_max, candidate)
        adjusted[idx] = min(running_max, 1.0)

    return adjusted


omnibus_rows = []
pairwise_rows = []

for k in SELECTED_ARCHETYPES:
    cur = support_df[support_df["archetype"] == k]
    groups = {
        condition: cur.loc[cur["condition"] == condition, SUPPORT_METRIC].to_numpy(dtype=float)
        for condition in CONDITION_ORDER
    }

    stat, p = kruskal(*[groups[c] for c in CONDITION_ORDER])
    omnibus_rows.append({
        "archetype": k,
        "kruskal_stat": stat,
        "p_raw": p,
    })

    local_rows = []
    for a, b in itertools.combinations(CONDITION_ORDER, 2):
        x = groups[a]
        y = groups[b]
        stat, p = mannwhitneyu(x, y, alternative="two-sided")

        # Rank-biserial effect size for independent samples.
        rbc = 2.0 * stat / (len(x) * len(y)) - 1.0

        local_rows.append({
            "archetype": k,
            "contrast": f"{a} - {b}",
            "n_a": len(x),
            "n_b": len(y),
            "median_difference": np.median(x) - np.median(y),
            "mannwhitney_u": stat,
            "rank_biserial": rbc,
            "p_raw": p,
        })

    adj = holm_adjust([row["p_raw"] for row in local_rows])
    for row, p_adj in zip(local_rows, adj):
        row["p_holm_within_archetype"] = p_adj

    pairwise_rows.extend(local_rows)

omnibus_df = pd.DataFrame(omnibus_rows)
omnibus_df["p_holm_across_archetypes"] = holm_adjust(
    omnibus_df["p_raw"].to_numpy()
)

pairwise_df = pd.DataFrame(pairwise_rows)

print("Omnibus independent-group condition tests:")
display(omnibus_df)

print("\nPairwise independent-group contrasts:")
display(pairwise_df)


## 6. Sensitivity metrics

In [ ]:

sensitivity_summary = (
    support_df
    .groupby(["archetype", "condition"], as_index=False)
    .agg(
        mean_support_l1=("support_l1", "mean"),
        mean_support_l2=("support_l2", "mean"),
        mean_support_relative=("support_relative", "mean"),
        mean_component_fro=("component_fro", "mean"),
        mean_loo_map_cosine=("loo_map_cosine", "mean"),
    )
)

display(sensitivity_summary)



`support_l2` remains the requested primary spatial-support measure. The additional columns answer slightly different questions:

- `support_l1`: total spatial allocation to the archetype across nodes;
- `support_relative`: L2 support relative to the complete coefficient matrix for that participant;
- `component_fro`: exact Frobenius magnitude of that archetype's rank-1 reconstructed contribution;
- `loo_map_cosine`: whether the participant's spatial map resembles the map recovered from other participants in the same condition.

Agreement across these summaries would make the broad-support conclusion much stronger than norm magnitude alone.


## 7. Export tables and figures

In [ ]:

network_mass_df.to_csv(
    OUTPUT_DIR / "selected_archetype_network_mass.csv",
    index=False,
)
support_df.to_csv(
    OUTPUT_DIR / "selected_archetype_support_by_independent_participant.csv",
    index=False,
)
support_summary_df.to_csv(
    OUTPUT_DIR / "selected_archetype_support_breadth_independent.csv",
    index=False,
)
map_similarity_summary.to_csv(
    OUTPUT_DIR / "selected_archetype_map_similarity_by_condition.csv",
    index=False,
)
condition_summary_df.to_csv(
    OUTPUT_DIR / "selected_archetype_condition_summary_independent.csv",
    index=False,
)
omnibus_df.to_csv(
    OUTPUT_DIR / "selected_archetype_condition_kruskal.csv",
    index=False,
)
pairwise_df.to_csv(
    OUTPUT_DIR / "selected_archetype_condition_pairwise_mannwhitney.csv",
    index=False,
)

print("Saved analysis outputs to:", OUTPUT_DIR.resolve())



## 8. Interpretation guardrails

### What the main figure can support
> At the representative intermediate-resolution solution (\(K=46\)), cross-fitted sparse selection identified five reproducibly informative archetypes. Spatial-AA archetypes were visualized as mean temporal profiles together with participant-averaged node-wise coefficient maps and Yeo-7 network composition.

### Broad participant support
The strongest evidence is the combination of:
- high within-condition `effective_fraction`;
- low `max_share`;
- high leave-one-out spatial-map cosine similarity.

This distinguishes **broad magnitude support** from **spatial-map reproducibility**.

### Condition dependence
The independent-group tests ask whether gross archetype expression differs across experimental groups. A lack of a support-magnitude difference does **not** imply that an archetype is equally informative for decoding in all conditions; decoding depends on the structure and cross-participant alignment of the reconstructed signal, not only on coefficient magnitude.

### Network interpretation
Yeo-7 bars are descriptive composition summaries. Claims of network *enrichment* should still rely on the previously developed network-size-preserving null analysis rather than raw mass alone.
